# Maestra `compare()` quickstart

**Not executed in CI** (offline smoke-test coverage instead: `tests/test_public_api.py`, two sklearn dummies). Run this notebook interactively — in Colab or locally — with real internet/API access.

`compare()` is Maestra's arbiter as a generic DS tool: it honestly decides whether one sklearn-compatible pipeline measurably beats another, using the same paired, Nadeau-Bengio-corrected statistical test every internal Maestra gate uses (`docs/RESULTS.md`'s N1 hardening). No LLM call, and — deliberately — **no AutoGluon needed**: `compare()` only touches plain sklearn estimators.

Because this notebook never needs AutoGluon (a large, slow install pulling in PyTorch), we install Maestra with `--no-deps` below and add only the light dependencies `compare()` actually needs. Everything else in Maestra's `pyproject.toml` (AutoGluon, litellm, ...) stays uninstalled — this notebook's whole point is that `compare()` doesn't need them.

In [ ]:
# Colab already ships pandas/numpy/scikit-learn. --no-deps skips maestra's own
# pyproject.toml dependencies (in particular autogluon.tabular[all], which pulls in PyTorch)
# since compare() never touches them.
!pip install --no-deps -q git+https://github.com/helenaschulz/maestra.git

In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from maestra import compare

# A small, built-in regression dataset (442 rows) -- no download, no API key.
data = load_diabetes(as_frame=True)
df = data.frame  # features + "target" column, already a single DataFrame
df.head()

In [ ]:
# Two arbitrary sklearn-compatible PIPELINES -- compare() only needs fit/predict(/predict_proba),
# which sklearn's own Pipeline already provides. Neither is AutoGluon; both are just sklearn.
baseline = LinearRegression()
candidate = Pipeline([
    ("scale", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge(alpha=1.0)),
])

result = compare(baseline, candidate, df, "target", cv=5, seeds=3)
print(result.summary())

`result.verdict` is one of `"improved"`, `"no_improvement"`, or `"underpowered"` — never a bare
number to eyeball. `result.mde` reports the minimum effect size this many seeds/folds could
even have detected, so an `"underpowered"`/`"no_improvement"` verdict is distinguishable from
"no effect" (see `docs/RESULTS.md`'s P0.2 MDE-Ausweis). `result.deltas` holds every pooled,
paired per-(seed, fold) delta if you want to inspect the spread yourself:

In [ ]:
pd.Series(result.deltas, name="paired delta (candidate - baseline)").describe()